Set reproducible CPU-only loaders, apply training augmentation, and load the measured Day 9 validation results for the final model comparison.

In [ ]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "models.py").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Could not locate project src/models.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.datasets import CsvImageDataset
from src.models import RegularizedCNN
from src.train_utils import evaluate, fit
from src.transforms import get_transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(torch.get_num_threads(), 2))
DEVICE = torch.device("cpu")
DATA_ROOT = PROJECT_ROOT / "data" / "catdog"
baseline_path = DATA_ROOT / "week2_baseline_metrics.json"
if not baseline_path.is_file():
    raise FileNotFoundError("Run the Day 9 notebook first to produce baseline metrics.")
baseline_metrics = json.loads(baseline_path.read_text(encoding="utf-8"))

stats_dataset = CsvImageDataset(DATA_ROOT / "train.csv", transform=transforms.ToTensor())
stats_loader = DataLoader(stats_dataset, batch_size=256, shuffle=False, num_workers=0)
channel_sum = torch.zeros(3)
channel_squared_sum = torch.zeros(3)
pixel_count = 0
for images, _ in stats_loader:
    channel_sum += images.sum(dim=(0, 2, 3))
    channel_squared_sum += (images ** 2).sum(dim=(0, 2, 3))
    pixel_count += images.size(0) * images.size(2) * images.size(3)
mean = (channel_sum / pixel_count).tolist()
std = torch.sqrt(channel_squared_sum / pixel_count - (channel_sum / pixel_count) ** 2).tolist()

train_dataset = CsvImageDataset(DATA_ROOT / "train.csv", transform=get_transforms(True, mean, std))
val_dataset = CsvImageDataset(DATA_ROOT / "val.csv", transform=get_transforms(False, mean, std))
test_dataset = CsvImageDataset(DATA_ROOT / "test.csv", transform=get_transforms(False, mean, std))
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)
print(f"Train: {len(train_dataset):,}; validation: {len(val_dataset):,}; test: {len(test_dataset):,}; device: {DEVICE}")


Train RegularizedCNN with augmentation for up to 30 epochs and stop after five validation-loss epochs without improvement, restoring the best weights.

In [ ]:
torch.manual_seed(SEED)
model = RegularizedCNN().to(DEVICE)
history = fit(
    model,
    train_loader,
    val_loader,
    epochs=30,
    lr=1e-3,
    patience=5,
)
best_epoch = history["best_epoch"]
print(f"Best epoch: {best_epoch}; best validation loss: {history['best_val_loss']:.4f}")
for epoch, (train_loss, train_acc, val_loss, val_acc) in enumerate(
    zip(history["train_loss"], history["train_acc"], history["val_loss"], history["val_acc"]), start=1
):
    print(
        f"epoch {epoch:02d} | train loss {train_loss:.4f} | train acc {train_acc:.4f} | "
        f"val loss {val_loss:.4f} | val acc {val_acc:.4f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
epochs_seen = range(1, len(history["train_loss"]) + 1)
axes[0].plot(epochs_seen, history["train_loss"], label="Train loss")
axes[0].plot(epochs_seen, history["val_loss"], label="Validation loss")
axes[0].set_title("Loss")
axes[1].plot(epochs_seen, history["train_acc"], label="Train accuracy")
axes[1].plot(epochs_seen, history["val_acc"], label="Validation accuracy")
axes[1].set_title("Accuracy")
for axis in axes:
    axis.axvline(best_epoch, color="black", linestyle="--", label=f"Best epoch: {best_epoch}")
    axis.set_xlabel("Epoch")
    axis.grid(alpha=0.3)
    axis.legend()
fig.tight_layout()
plt.show()

checkpoint_path = PROJECT_ROOT / "models" / "regularized_cnn_catdog.pth"
if checkpoint_path.exists():
    raise FileExistsError(f"Refusing to overwrite existing weights: {checkpoint_path}")
torch.save(model.state_dict(), checkpoint_path)
print("Saved best weights to", checkpoint_path)


Evaluate the held-out test split exactly once, report classification metrics and ROC-AUC, plot the ROC curve and eight mistakes, then compare model sizes and train-validation gaps.

In [ ]:
criterion = nn.CrossEntropyLoss()
test_loss, test_accuracy, test_probabilities = evaluate(model, test_loader, criterion)
test_labels = np.asarray([label for _, label in ImageFolder(DATA_ROOT / "test").samples])
test_scores = test_probabilities[:, 1].numpy()
test_predictions = test_probabilities.argmax(dim=1).numpy()
assert len(test_labels) == len(test_predictions)
class_names = ImageFolder(DATA_ROOT / "test").classes
roc_auc = roc_auc_score(test_labels, test_scores)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test ROC-AUC: {roc_auc:.4f}")
print("Classification report:")
print(classification_report(test_labels, test_predictions, target_names=class_names, digits=4, zero_division=0))
print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(test_labels, test_predictions, labels=[0, 1]))

false_positive_rate, true_positive_rate, _ = roc_curve(test_labels, test_scores)
fig, axis = plt.subplots(figsize=(6, 5))
axis.plot(false_positive_rate, true_positive_rate, label=f"RegularizedCNN (AUC = {roc_auc:.4f})")
axis.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
axis.set(xlabel="False positive rate", ylabel="True positive rate", title="Test ROC curve")
axis.legend()
axis.grid(alpha=0.3)
fig.tight_layout()
plt.show()

misclassified = np.flatnonzero(test_predictions != test_labels)
if len(misclassified) < 8:
    raise RuntimeError(f"Only {len(misclassified)} test errors; cannot display the requested eight examples.")
test_imagefolder = ImageFolder(DATA_ROOT / "test", transform=get_transforms(False, mean, std))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis, index in zip(axes.flat, misclassified[:8]):
    image, _ = test_imagefolder[int(index)]
    axis.imshow((image * torch.tensor(std).view(3, 1, 1) + torch.tensor(mean).view(3, 1, 1)).clamp(0, 1).permute(1, 2, 0).numpy())
    axis.set_title(f"True: {class_names[test_labels[index]]}\nPred: {class_names[test_predictions[index]]}")
    axis.axis("off")
fig.suptitle("Eight misclassified test images")
fig.tight_layout()
plt.show()

best_index = best_epoch - 1
regularized_summary = {
    "model": "RegularizedCNN (aug, early stopping)",
    "val_acc": float(history["val_acc"][best_index]),
    "test_acc": float(test_accuracy),
    "parameters": sum(parameter.numel() for parameter in model.parameters()),
    "train_val_gap": float(history["train_acc"][best_index] - history["val_acc"][best_index]),
}
comparison = pd.DataFrame(baseline_metrics + [regularized_summary])
comparison["val_acc"] = comparison["val_acc"].map(lambda value: f"{value:.4f}")
comparison["test_acc"] = comparison["test_acc"].map(lambda value: "—" if pd.isna(value) else f"{value:.4f}")
comparison["train_val_gap"] = comparison["train_val_gap"].map(lambda value: f"{value:.4f}")
display(comparison)
print(f"Best epoch: {best_epoch}; test ROC-AUC: {roc_auc:.4f}")
